# Clustering Analysis of Colorado Bird Sightings

This notebook uses clustering analysis on `birdsong_df` to answer the following research questions:

9. How has the prevalence of different species of birds changed over time?
10. What are the typical flight patterns of different bird species?  

Since observations will be clustered in different ways across both questions, separate models are built for each.  
When identifying bird species, we refer to the family that the species belongs to.  

## Why K-Means Clustering Was Chosen

`birdsong_df` is largely unlabelled for the abovementioned questions. There is no ground-truth label that says which bird species belong to the same temporal prevalence profile or which species share similar flight behavior. That makes unsupervised learning the right modeling family.

K-Means is used here because:

- The transformed feature sets are numeric and can be standardized.
- We want interpretable centroids that summarize each cluster.
- The dataset is large enough that a fast centroid-based method is practical.
- We can tune the number of clusters using internal validation metrics.

## Model Assumptions

K-Means assumes:

- Clusters are reasonably compact and separable in feature space.
- Euclidean distance is meaningful after feature scaling.
- Features with larger raw units should not dominate, so scaling is required.
- The chosen number of clusters `k` is not known in advance and must be tuned.

For the flight-pattern analysis, an additional practical assumption is made: because the dataset contains observations rather than true tracked trajectories, flight patterns are approximated using seasonal and monthly geographic movement signatures rather than literal path reconstruction.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='talk')

In [2]:
birdsong_df = pd.read_csv('..\\data\\birdsong.csv')
birdsong_df

,common_name,date,bird_count,county,family,total_monthly_precipitation,monthly_max_temp,monthly_min_temp,urban_population_level,county_total_population,is_fire,is_flood,bird_count_adjusted_for_population,season
0,Band-tailed Pigeon,2021-01,1.0000,Garfield,Pigeons and Doves,23.0158,1.4122,-10.4589,4,"62,178.4800",0,0,0.0000,Winter
1,Pine Warbler,2021-01,1.0000,Boulder,New World Warblers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
2,White-winged Scoter,2021-01,1.0000,Larimer,"Ducks, Geese, and Waterfowl",10.8259,1.8541,-10.4572,5,"361,938.5280",0,0,0.0000,Winter
3,Brown Thrasher,2021-01,1.0000,Boulder,Mockingbirds and Thrashers,9.8737,3.8300,-8.2778,5,"333,404.0640",0,0,0.0000,Winter
4,Bonaparte's Gull,2021-01,2.0000,Pueblo,"Gulls, Terns, and Skimmers",14.1278,8.5667,-8.4000,5,"169,507.2960",0,0,0.0000,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349425,Sandhill Crane,2025-12,1.0000,Weld,Cranes,6.6353,12.0575,-5.7237,4,"342,352.4790",0,0,0.0000,Winter
349426,Scaled Quail,2025-12,3.0000,Prowers,New World Quail,2.1875,14.2700,-6.2800,4,"12,486.7010",0,0,0.0002,Winter
349427,Brown-headed Cowbird,2025-12,1.0000,Boulder,Troupials and Allies,19.5974,9.1456,-2.9856,5,"344,201.7054",0,0,0.0000,Winter
349428,White-throated Sparrow,2025-12,1.0000,Larimer,New World Sparrows,17.0463,5.9413,-4.6144,5,"373,660.2880",0,0,0.0000,Winter


## Hyperparameter Tuning

Hyperparameter tuning is handled by testing multiple values of `k` and selecting the value that maximizes Silhouette Score, with Davies-Bouldin Index used as a secondary quality check.

In [3]:
# Test over multiple values of k to find the one that gives the best evaluation score
def evaluate_kmeans_grid(X, k_values=range(2, 9), random_state=42):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        rows.append({
            'k': k,
            'silhouette_score': silhouette_score(X, labels),
            'davies_bouldin_index': davies_bouldin_score(X, labels)
        })
    scores = pd.DataFrame(rows)
    best_k = scores.sort_values(['silhouette_score', 'davies_bouldin_index'], ascending=[False, True]).iloc[0]['k']
    return scores, int(best_k)

# Create a model with the best k value and fit the data to it
def fit_final_kmeans(X, best_k, random_state=42):
    model = KMeans(n_clusters=best_k, random_state=random_state, n_init=20)
    labels = model.fit_predict(X)  # cluster labels are numbers 1, 2, ..., k
    metrics = {
        'silhouette_score': silhouette_score(X, labels),
        'davies_bouldin_index': davies_bouldin_score(X, labels)
    }
    return model, labels, metrics

## 9. Bird Species Prevalence Changes Over Time

To study how bird species prevalence changes over time, each bird family is represented by a monthly prevalence profile. Prevalence is defined as total observed count per month, and a small `log1p` transform is applied to reduce the influence of unusually large counts.

In [4]:
family_monthly = (
    birdsong_df
    .groupby(['family', 'date'], as_index=False)
    .agg(monthly_bird_count=('bird_count', 'sum'))
)
family_monthly['log_monthly_bird_count'] = np.log1p(family_monthly['monthly_bird_count'])

### Snapshot Before Prevalence Pivot

In [5]:
family_monthly.head(10)

,family,date,monthly_bird_count,log_monthly_bird_count
0,Anhingas,2021-05,1.0000,0.6931
1,Anhingas,2025-06,23.0000,3.1781
2,Anhingas,2025-07,26.0000,3.2958
3,Anhingas,2025-08,10.0000,2.3979
4,Barn-Owls,2021-01,10.0000,2.3979
5,Barn-Owls,2021-02,5.0000,1.7918
6,Barn-Owls,2021-03,4.0000,1.6094
7,Barn-Owls,2021-04,14.0000,2.7081
8,Barn-Owls,2021-05,22.0000,3.1355
9,Barn-Owls,2021-06,13.0000,2.6391


### Data Transformation Steps

Reshape the family prevalence data into a format that can be clustered.

- `family_prevalence_wide` pivots the monthly prevalence table so that each row represents one bird family and each column represents one month in the time series. The values are the log-transformed monthly bird counts. This turns each family into a time-series feature vector that captures how its prevalence changes over time.
- `.fillna(0)` replaces missing family-month combinations with `0`, meaning the family had no recorded prevalence in that month. This is necessary because clustering requires a complete numeric matrix.
- `family_prevalence_scaled` applies `StandardScaler()` to the pivoted matrix so the monthly columns are on a comparable scale. In the clustering analysis, this prevents months with larger raw count ranges from dominating the distance calculations and helps the model group families by the shape of their prevalence pattern rather than just absolute magnitude.

Together, these transformations produce a clean family-by-time matrix that is suitable for K-Means clustering of prevalence trends.

In [6]:
family_prevalence_wide = (
    family_monthly
    .pivot(index='family', columns='date', values='log_monthly_bird_count')
    .fillna(0)
)

family_prevalence_scaled = pd.DataFrame(
    StandardScaler().fit_transform(family_prevalence_wide),
    index=family_prevalence_wide.index,
    columns=family_prevalence_wide.columns
)

### Snapshot After Prevalence Scaling

In [7]:
family_prevalence_scaled

date,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
family,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Anhingas,-1.2794,-1.2593,-1.5275,-1.8403,-1.7401,-2.0390,-2.0480,-1.9744,-1.9559,-1.7639,-1.4047,-1.2877,-1.2731,-1.2530,-1.5237,-1.8812,-2.0324,-2.0807,-2.0348,-1.9269,-2.0288,-1.8327,-1.5172,-1.2829,-1.2362,-1.2758,-1.4620,-2.0045,-2.1095,-2.3164,-2.0960,-2.0139,-2.0064,-1.8758,-1.4567,-1.3381,-1.2954,-1.3597,-1.6158,-1.9543,-2.1101,-2.1493,-2.0672,-2.1770,-2.0797,-1.9106,-1.5415,-1.3707,-1.2876,-1.3297,-1.5730,-1.9279,-1.9344,-0.5681,-0.6429,-1.1489,-2.0549,-1.8534,-1.5747,-1.3420
Barn-Owls,-0.4574,-0.6321,-0.9219,-0.7194,-0.6335,-0.8255,-1.1160,-0.8395,-1.0487,-1.0530,-1.1516,-1.0451,-0.8923,-1.2530,-0.6571,-0.7013,-0.6920,-0.6327,-0.7090,-0.2420,-0.6094,-1.0542,-1.0944,-0.8093,-0.5120,-1.2758,-0.3619,-0.7066,-0.6120,-0.5538,-0.5508,-0.7407,-0.5939,-1.1135,-0.9457,-0.4475,-0.4060,-0.4602,-0.5281,-0.5752,-0.6871,-0.4113,-0.7262,-0.3466,-0.3153,-0.7840,-1.1075,-0.4537,-0.5812,-0.4451,-0.3255,-0.5903,-0.6639,-0.6120,-0.7402,-0.5080,-0.5205,-0.9049,-0.8661,-0.5286
Boobies and Gannets,-1.2794,-1.2593,-1.5275,-1.8403,-2.0541,-2.0390,-2.0480,-1.9744,-1.9559,-1.7639,-1.4047,-1.2877,-1.2731,-1.2530,-1.5237,-1.8812,-2.0324,-2.0807,-2.0348,-1.9269,-2.0288,-1.8327,-1.5172,-1.2829,-1.2362,-1.2758,-1.4620,-2.0045,-2.1095,-2.3164,-2.0960,-2.0139,-2.0064,-1.8758,-1.4567,-1.3381,-1.2954,-1.3597,-1.6158,-1.9543,-2.1101,-2.1493,-2.0672,-2.1770,-2.0797,-1.9106,-1.5415,-1.3707,-1.2876,-1.3297,-1.5730,-1.9279,-1.9344,-1.4720,-2.2086,-2.2959,-2.0549,-1.8534,-1.5747,-1.3420
Cardinals and Allies,-0.3289,-0.0572,-0.4002,-0.0999,0.7178,0.6572,0.5539,0.4075,0.3052,-0.3206,-0.6942,-0.6067,-0.3121,0.0867,-0.2194,0.0282,0.7225,0.6844,0.5296,0.3944,0.3341,-0.1752,-0.3456,-0.1571,-0.1888,-0.1289,-0.1262,0.0120,0.7125,0.7537,0.6953,0.4819,0.3636,-0.2287,-0.3912,-0.5506,-0.2931,-0.0838,-0.1443,0.1289,0.6246,0.7110,0.6034,0.3550,0.2598,-0.1922,-0.1259,-0.3180,-0.3058,-0.1573,-0.3255,-0.0122,0.7205,0.7094,0.7142,0.4766,0.2759,-0.0860,-0.8051,-0.3412
Cassowaries and Emu,-1.2794,-1.2593,-1.5275,-1.8403,-2.0541,-2.0390,-2.0480,-1.9744,-1.9559,-1.7639,-1.4047,-1.2877,-1.2731,-1.2530,-1.5237,-1.8812,-2.0324,-2.0807,-2.0348,-1.9269,-2.0288,-1.8327,-1.5172,-1.2829,-1.2362,-1.2758,-1.4620,-2.0045,-2.1095,-2.3164,-2.0960,-2.0139,-2.0064,-1.8758,-1.4567,-1.3381,-1.2954,-1.3597,-1.6158,-1.9543,-2.1101,-2.1493,-2.0672,-2.1770,-2.0797,-1.9106,-1.5415,-1.3707,-1.2876,-1.3297,-1.5730,-1.9279,-1.9344,-2.1713,-1.8793,-2.2959,-2.0549,-1.8534,-1.5747,-1.3420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wagtails and Pipits,0.2354,0.2567,0.2055,0.3444,0.1261,0.0739,0.1669,-0.0403,-0.1518,0.3138,0.2343,0.2544,0.1738,0.1474,-0.0910,-0.1831,0.2785,-0.0107,0.0915,0.1671,0.0294,0.3340,0.4704,0.2936,0.1997,0.1710,0.0354,-0.0567,0.0897,0.5177,0.0476,-0.1515,0.1631,0.2417,0.6219,0.4808,0.2198,0.1226,0.2900,-0.0324,0.4337,0.1260,0.0489,0.1835,-0.1906,0.2991,0.1993,0.1217,0.2933,0.4302,0.2631,-0.2088,0.4439,-0.0502,-0.1037,0.1774,-0.3704,0.1012,0.2583,0.6989
Waxwings,0.9186,0.8415,0.7009,0.5052,0.3036,0.0598,-0.1250,-0.0572,0.2317,0.3383,0.5131,0.6688,0.8270,0.8618,0.6842,0.3994,0.0868,-0.1110,-0.0901,0.0504,0.1981,0.1414,0.5671,1.0606,1.1897,1.3845,1.4847,1.0211,0.3315,0.1366,-0.2066,0.0614,0.2876,0.4248,0.6179,1.0397,0.7950,0.8432,0.8580,0.5304,0.2235

### Applying K-Means Clustering Models

K-means clustering is done for values of `k` from 2 to 9. For each value of `k`, the Silhouette Score and Davies Bouldin Index are computed.

In [8]:
prevalence_scores, prevalence_best_k = evaluate_kmeans_grid(family_prevalence_scaled, k_values=range(2, 9))
prevalence_model, prevalence_labels, prevalence_metrics = fit_final_kmeans(family_prevalence_scaled, prevalence_best_k)

prevalence_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.5151,0.7198
1,3,0.4799,0.7712
2,4,0.3966,0.8493
3,5,0.4046,0.9232
4,6,0.4102,0.8394
5,7,0.3747,0.9379
6,8,0.3610,0.9200


The model that uses the `k` value with the best scores is used to generate the clusters.  
Then, each observation is assigned its corresponding cluster label.  
Finally, the following are computed for each cluster:

1. Number of families in the cluster
2. Average standardized prevalence of the families in the cluster
3. Average time trend for the families in the cluster: Generally increasing, decreasing, or stable over time

In [9]:
family_prevalence_clusters = family_prevalence_scaled.copy()
family_prevalence_clusters['cluster'] = prevalence_labels
family_prevalence_clusters['avg_scaled_prevalence'] = family_prevalence_scaled.mean(axis=1)
family_prevalence_clusters['prevalence_trend'] = (
    family_prevalence_wide.iloc[:, -12:].mean(axis=1) - family_prevalence_wide.iloc[:, :12].mean(axis=1)
)

prevalence_cluster_summary = (
    family_prevalence_clusters
    .groupby('cluster')
    .agg(
        family_count=('avg_scaled_prevalence', 'size'),
        mean_scaled_prevalence=('avg_scaled_prevalence', 'mean'),
        mean_trend=('prevalence_trend', 'mean')
    )
    .sort_values('mean_trend', ascending=False)
)

prevalence_cluster_summary

,family_count,mean_scaled_prevalence,mean_trend
cluster,,,
1,18,-1.2277,0.1539
0,47,0.4702,-0.1419


In [10]:
prevalence_example_families = (
    family_prevalence_clusters
    .reset_index()
    .rename(columns={'index': 'family'})
    .sort_values(['cluster', 'avg_scaled_prevalence', 'prevalence_trend'], ascending=[True, False, False])
    .groupby('cluster')['family']
    .apply(lambda s: list(s.head(5)))
    .to_dict()
)

prevalence_cluster_lines = ['### Cluster Interpretation', '']

for cluster_num, summary_row in prevalence_cluster_summary.sort_index().iterrows():
    display_cluster_num = int(cluster_num) + 1
    trend_direction = 'positive' if summary_row['mean_trend'] > 0 else 'negative' if summary_row['mean_trend'] < 0 else 'near-zero'
    trend_text = (
        'a broadly increasing prevalence profile over the period from 2021 to 2025'
        if summary_row['mean_trend'] > 0
        else 'a slight softening in prevalence over the period from 2021 to 2025'
        if summary_row['mean_trend'] < 0
        else 'a relatively stable prevalence profile over the period from 2021 to 2025'
    )
    prevalence_text = (
        'These families are relatively lower-prevalence families overall.'
        if summary_row['mean_scaled_prevalence'] < 0
        else 'These families remain comparatively common overall.'
        if summary_row['mean_scaled_prevalence'] > 0
        else 'These families sit near the overall average prevalence level.'
    )
    examples = ', '.join(f'**{family}**' for family in prevalence_example_families.get(cluster_num, []))
    prevalence_cluster_lines.append(
        f"- **Cluster {display_cluster_num}:** This cluster contains **{int(summary_row['family_count'])} families** and has a {trend_direction} average `mean_trend`, which suggests {trend_text}. Its {'negative' if summary_row['mean_scaled_prevalence'] < 0 else 'positive' if summary_row['mean_scaled_prevalence'] > 0 else 'near-zero'} `mean_scaled_prevalence` indicates that {prevalence_text} Example families in this cluster include {examples}."
    )

display(Markdown('\n'.join(prevalence_cluster_lines)))

### Cluster Interpretation

- **Cluster 1:** This cluster contains **47 families** and has a negative average `mean_trend`, which suggests a slight softening in prevalence over the period from 2021 to 2025. Its positive `mean_scaled_prevalence` indicates that These families remain comparatively common overall. Example families in this cluster include **Ducks, Geese, and Waterfowl**, **Gulls, Terns, and Skimmers**, **Troupials and Allies**, **Crows, Jays, and Magpies**, **Finches, Euphonias, and Allies**.
- **Cluster 2:** This cluster contains **18 families** and has a positive average `mean_trend`, which suggests a broadly increasing prevalence profile over the period from 2021 to 2025. Its negative `mean_scaled_prevalence` indicates that These families are relatively lower-prevalence families overall. Example families in this cluster include **Hummingbirds**, **Vireos, Shrike-Babblers, and Erpornis**, **Osprey**, **Gnatcatchers**, **Cuckoos**.

## 10. Typical Flight Patterns by Bird Speices

This analysis estimates flight-pattern types using the county locations recorded in `birdsong_df` across time. Each family of species is represented by its county-level distribution over months. That lets the model group families with similar geographic spread, county turnover, and seasonal concentration patterns. To turn those clusters into interpretable movement corridors, county centroids are estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then aggregated into month-to-month cluster paths.

In [11]:
birdsong_df['month'] = pd.to_datetime(birdsong_df['date'].astype(str)).dt.month

family_monthly_county = (
    birdsong_df
    .groupby(['family', 'month', 'county'], as_index=False)
    .agg(monthly_count=('bird_count', 'sum'))
)

family_monthly_county['log_monthly_count'] = np.log1p(family_monthly_county['monthly_count'])

### Snapshot Before Flight-Pattern Feature Engineering

In [12]:
family_monthly_county.head(10)

,family,month,county,monthly_count,log_monthly_count
0,Anhingas,5,Boulder,1.0000,0.6931
1,Anhingas,6,Boulder,23.0000,3.1781
2,Anhingas,7,Boulder,26.0000,3.2958
3,Anhingas,8,Boulder,10.0000,2.3979
4,Barn-Owls,1,Adams,13.0000,2.6391
5,Barn-Owls,1,Arapahoe,9.0000,2.3026
6,Barn-Owls,1,Bent,1.0000,0.6931
7,Barn-Owls,1,Boulder,1.0000,0.6931
8,Barn-Owls,1,El Paso,1.0000,0.6931
9,Barn-Owls,1,Kit Carson,2.0000,1.0986


### Feature Engineering Steps

This block converts county-by-month bird observations into features that can be clustered into typical flight-pattern types.

- `family_county_month_wide` creates a wide matrix where each row is a bird family and each column is a specific month-county combination such as `m03_Boulder`. The values are log-transformed bird counts. In the clustering analysis, this captures where a family tends to appear and when it appears there.
- `monthly_family_summary` summarizes each family by month. It keeps track of total monthly count and the number of active counties. In the clustering analysis, this helps describe how broadly a family is distributed in a typical month.
- `county_turnover` converts each family-month into a set of counties. That makes it possible to compare one month's county footprint with the next month's footprint.
- The `turnover_rows` loop calculates `avg_county_turnover`, which measures how much a family changes counties from month to month. Values closer to `0` mean the family tends to stay in the same counties, while larger values indicate stronger geographic shifting over time.
- `flight_pattern_features` combines several summary traits for each family: how many counties it visits, how many months it is active, its average count, its average monthly county spread, and its month-to-month county turnover. These are the compact behavioral features used to describe movement style.
- `log_total_observed_count` is added to reduce the effect of extremely large raw counts. This keeps very common families from dominating the clustering simply because of scale rather than movement behavior.

Together, these transformations give the clustering model both a detailed month-county distribution matrix and a smaller set of summary movement features, so clusters reflect movement behavior rather than just raw abundance.

In [13]:
family_county_month_wide = (
    family_monthly_county
    .assign(month_county=lambda df: 'm' + df['month'].astype(str).str.zfill(2) + '_' + df['county'].str.replace(' ', '_', regex=False))
    .pivot(index='family', columns='month_county', values='log_monthly_count')
    .fillna(0)
)

monthly_family_summary = (
    family_monthly_county
    .groupby(['family', 'month'], as_index=False)
    .agg(
        monthly_total_count=('monthly_count', 'sum'),
        counties_active=('county', 'nunique')
    )
)

county_turnover = (
    family_monthly_county
    .groupby(['family', 'month'])['county']
    .apply(lambda x: set(x))
    .reset_index(name='county_set')
    .sort_values(['family', 'month'])
)

turnover_rows = []
for family_name, family_df in county_turnover.groupby('family'):
    sets = list(family_df['county_set'])
    if len(sets) <= 1:
        turnover_rows.append({'family': family_name, 'avg_county_turnover': 0.0})
        continue

    turnovers = []
    for prev_set, curr_set in zip(sets[:-1], sets[1:]):
        union_size = len(prev_set | curr_set)
        if union_size == 0:
            turnovers.append(0.0)
        else:
            turnovers.append(1 - (len(prev_set & curr_set) / union_size))
    turnover_rows.append({'family': family_name, 'avg_county_turnover': float(np.mean(turnovers))})

county_turnover_features = pd.DataFrame(turnover_rows)

flight_pattern_features = (
    family_monthly_county
    .groupby('family', as_index=False)
    .agg(
        counties_visited=('county', 'nunique'),
        active_months=('month', 'nunique'),
        avg_monthly_count=('monthly_count', 'mean'),
        total_observed_count=('monthly_count', 'sum')
    )
    .merge(
        monthly_family_summary.groupby('family', as_index=False).agg(avg_active_counties=('counties_active', 'mean')),
        on='family',
        how='left'
    )
    .merge(county_turnover_features, on='family', how='left')
)

flight_pattern_features['log_total_observed_count'] = np.log1p(flight_pattern_features['total_observed_count'])
flight_pattern_features = flight_pattern_features.drop(columns='total_observed_count')

Next, select summary movement features and standardize their values by applying `StandardScaler()` to transform them onto comparable scale to prevent larger values from dominating the distance calculations.  
Additionally, standardize the values in the matrix of families and the month-county combinations.  
Then, join them together.

In [14]:
flight_numeric_cols = [
    'counties_visited', 'active_months', 'avg_monthly_count',
    'avg_active_counties', 'avg_county_turnover', 'log_total_observed_count'
]

flight_summary_scaled = pd.DataFrame(
    StandardScaler().fit_transform(flight_pattern_features.set_index('family')[flight_numeric_cols]),
    index=flight_pattern_features['family'],
    columns=flight_numeric_cols
)

flight_distribution_scaled = pd.DataFrame(
    StandardScaler().fit_transform(family_county_month_wide),
    index=family_county_month_wide.index,
    columns=family_county_month_wide.columns
)

flight_pattern_scaled = flight_summary_scaled.join(flight_distribution_scaled, how='inner')

### Snapshot After Flight-Pattern Feature Engineering

In [15]:
flight_pattern_scaled.head(10)

,counties_visited,active_months,avg_monthly_count,avg_active_counties,avg_county_turnover,log_total_observed_count,m01_Adams,m01_Alamosa,m01_Arapahoe,m01_Archuleta,m01_Baca,m01_Bent,m01_Boulder,m01_Broomfield,m01_Chaffee,m01_Cheyenne,m01_Clear_Creek,m01_Costilla,m01_Crowley,m01_Custer,m01_Delta,m01_Denver,m01_Dolores,m01_Douglas,m01_Eagle,m01_El_Paso,m01_Elbert,m01_Fremont,m01_Garfield,m01_Gilpin,m01_Grand,m01_Gunnison,m01_Hinsdale,m01_Huerfano,m01_Jackson,m01_Jefferson,m01_Kiowa,m01_Kit_Carson,m01_La_Plata,m01_Lake,m01_Larimer,m01_Las_Animas,m01_Lincoln,m01_Logan,m01_Mesa,m01_Mineral,m01_Moffat,m01_Montezuma,m01_Montrose,m01_Morgan,...,m12_Crowley,m12_Custer,m12_Delta,m12_Denver,m12_Dolores,m12_Douglas,m12_Eagle,m12_El_Paso,m12_Elbert,m12_Fremont,m12_Garfield,m12_Gilpin,m12_Grand,m12_Gunnison,m12_Huerfano,m12_Jackson,m12_Jefferson,m12_Kiowa,m12_Kit_Carson,m12_La_Plata,m12_Lake,m12_Larimer,m12_Las_Animas,m12_Lincoln,m12_Logan,m12_Mesa,m12_Mineral,m12_Moffat,m12_Montezuma,m12_Montrose,m12_Morgan,m12_Otero,m12_Ouray,m12_Park,m12_Phillips,m12_Pitkin,m12_Prowers,m12_Pueblo,m12_Rio_Blanco,m12_Rio_Grande,m12_Routt,m12_Saguache,m12_San_Juan,m12_San_Miguel,m12_Sedgwick,m12_Summit,m12_Teller,m12_Washington,m12_Weld,m12_Yuma
family,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Anhingas,-2.2647,-1.9310,-0.4009,-1.6654,-1.8489,-1.5002,-0.6166,-0.3548,-1.0212,-0.5864,-0.5630,-0.6406,-1.0417,-0.6535,-0.5869,-0.2334,-0.3489,-0.2948,-0.3057,-0.3222,-0.5763,-0.8807,-0.5158,-0.7542,-0.5403,-1.0623,-0.3600,-0.8119,-0.6804,-0.2851,-0.4914,-0.2625,-0.2081,-0.4435,-0.2898,-1.0816,-0.4627,-0.3189,-0.8261,-0.2205,-1.0522,-0.4546,-0.3547,-0.4934,-1.0546,-0.2100,-0.4247,-0.8748,-0.7306,-0.2518,...,-0.4152,-0.4060,-0.5989,-0.8750,-0.3560,-0.7683,-0.4647,-1.0864,-0.2805,-0.7644,-0.5335,-0.3128,-0.4339,-0.4199,-0.6182,-0.2402,-1.1515,-0.3762,-0.4254,-0.8808,-0.2463,-1.1287,-0.4729,-0.3920,-0.6412,-0.9771,-0.2055,-0.3797,-0.9060,-0.6725,-0.5916,-0.7071,-0.6208,-0.4804,-0.3321,-0.5946,-0.5684,-1.1744,-0.3130,-0.3263,-0.4275,-0.2667,-0.1575,-0.3884,-0.3223,-0.5073,-0.3966,-0.4029,-0.8845,-0.5730
Barn-Owls,-0.7090,0.4944,-0.4551,-0.8656,0.5972,-0.5518,0.8141,-0.3548,-0.0985,-0.5864,-0.5630,-0.2347,-0.7822,-0.6535,-0.5869,-0.2334,-0.3489,-0.2948,-0.3057,-0.3222,-0.5763,-0.8807,-0.5158,-0.7542,-0.5403,-0.7491,-0.3600,-0.8119,-0.6804,-0.2851,-0.4914,-0.2625,-0.2081,-0.4435,-0.2898,-1.0816,-0.4627,0.3229,-0.8261,-0.2205,-0.7982,-0.4546,-0.3547,-0.4934,-1.0546,-0.2100,-0.4247,-0.8748,-0.7306,-0.2518,...,-0.4152,-0.4060,-0.5989,-0.8750,-0.3560,-0.7683,-0.4647,-0.7676,0.6030,-0.7644,-0.5335,-0.3128,-0.4339,-0.4199,-0.6182,-0.2402,-1.1515,-0.3762,-0.4254,-0.8808,-0.2463,-0.7113,-0.4729,-0.3920,-0.2554,-0.4093,-0.2055,-0.3797,-0.9060,-0.6725,-0.2076,-0.2897,-0.6208,-0.4804,-0.3321,-0.5946,-0.5684,-0.8731,-0.3130,-0.3263,-0.4275,-0.2667,-0.1575,-0.3884,-0.3223,-0.5073,-0.3966,-0.4029,-0.3716,-0.5730
Boobies and Gannets,-2.2647,-2.8406,-0.4711,-1.6654,-1.8489,-2.4323,-0.6166,-0.3548,-1.0212,-0.5864,-0.5630,-0.6406,-1.0417,-0.6535,-0.5869,-0.2334,-0.3489,-0.2948,-0.3057,-0.3222,-0.5763,-0.8807,-0.5158,-0.7542,-0.5403,-1.0623,-0.3600,-0.8119,-0.6804,-0.2851,-0.4914,-0.2625,-0.2081,-0.4435,-0.2898,-1.0816,-0.4627,-0.3189,-0.8261,-0.2205,-1.0522,-0.4546,-0.3547,-0.4934,-1.0546,-0.2100,-0.4247,-0.8748,-0.7306,-0.2518,...,-0.4152,-0.4060,-0.5989,-0.8750,-0.3560,-0.7683,-0.4647,-1.0864,-0.2805,-0.7644,-0.5335,-0.3128,-0.4339,-0.4199,-0.6182,-0.2402,-1.1515,-0.3762,-0.4254,-0.8808,-0.2463,-1.1287,-0.4729,-0.3920,-0.6412,-0.9771,-0.2055,-0.3797,-0.9060,-0.6725,-0.5916,-0.7071,-0.6208,-0.4804,-0.3321,-0.5946,-0.5684,-1.1744,-0.3130,-0.3263,-0.4275,-0.2667,-0.1575,-0.3884,-0.3223,-0.5073,-0.3966,-0.4029,-0.8845,-0.5730
Cardinals and Allies,0.7494,0.4944,-0.3500,0.3543,0.0405,0.2354,0.1349,-0.3548,-0.7434,-0.5864,-0.5630,0.6459,-0.7822,-0.6535,-0.5869,-0.2334,-0.3489,-0.2948,-0.3057,-0.3222,-0.5763,-0.8807,-0.5158,-0.7542,-0.54

### Applying K-Means Clustering Models

K-means clustering is done for values of `k` from 2 to 9. For each value of `k`, the Silhouette Score and Davies Bouldin Index are computed.

In [16]:
flight_scores, flight_best_k = evaluate_kmeans_grid(flight_pattern_scaled[flight_numeric_cols], k_values=range(2, 9))
flight_model, flight_labels, flight_metrics = fit_final_kmeans(flight_pattern_scaled[flight_numeric_cols], flight_best_k)

flight_pattern_scaled['cluster'] = flight_labels
flight_scores

,k,silhouette_score,davies_bouldin_index
0,2,0.5985,0.7432
1,3,0.3549,0.9787
2,4,0.3845,0.8084
3,5,0.4277,0.6923
4,6,0.4403,0.5661
5,7,0.3647,0.7093
6,8,0.3714,0.6527


The model that uses the `k` value with the best scores is used to generate the clusters.  
Then, each observation is assigned its corresponding cluster label.

In [17]:
flight_cluster_summary = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .groupby('cluster')
    .agg(
        family_count=('family', 'size'),
        counties_visited=('counties_visited', 'mean'),
        active_months=('active_months', 'mean'),
        avg_active_counties=('avg_active_counties', 'mean'),
        avg_county_turnover=('avg_county_turnover', 'mean'),
        avg_monthly_count=('avg_monthly_count', 'mean')
    )
    .sort_values(['avg_county_turnover', 'counties_visited'], ascending=False)
)

cluster_example_families = (
    flight_pattern_features
    .assign(cluster=flight_labels)
    .sort_values(['cluster', 'active_months', 'counties_visited', 'avg_monthly_count'], ascending=[True, False, False, False])
    .groupby('cluster')['family']
    .apply(lambda s: list(s.head(5)))
    .to_dict()
)

### Interpreting Clusters In Terms Of Movement Behavior

The analysis below combines the clustering output with county-level corridor inference so each cluster can be interpreted as a movement type.  
We map counties to their respective centroids, then find the coordinates for monthly cluster centers which are computed as weighted averages of the centroids of the cluster's counties, where larger weights are assigned to counties with more bird observations. Then we compare the monthly cluster centers across time to identify the clusters' movement paths and overall directions.

In [18]:
raw_ebird_df = pd.read_csv('../data/ebird_co.csv')

county_centroids = (
    raw_ebird_df
    .dropna(subset=['subnational2Name', 'lat', 'lng'])
    .groupby('subnational2Name', as_index=False)
    .agg(
        county_lat=('lat', 'mean'),
        county_lng=('lng', 'mean')
    )
    .rename(columns={'subnational2Name': 'county'})
)

flight_cluster_assignments = flight_pattern_features[['family']].copy()
flight_cluster_assignments['cluster'] = flight_labels

cluster_monthly_county_paths = (
    family_monthly_county
    .merge(flight_cluster_assignments, on='family', how='inner')
    .merge(county_centroids, on='county', how='left')
    .dropna(subset=['county_lat', 'county_lng'])
    .groupby(['cluster', 'month', 'county', 'county_lat', 'county_lng'], as_index=False)
    .agg(cluster_monthly_count=('monthly_count', 'sum'))
)

cluster_monthly_centers = (
    cluster_monthly_county_paths
    .groupby(['cluster', 'month'], as_index=False)
    .apply(
        lambda df: pd.Series({
            'weighted_lat': np.average(df['county_lat'], weights=df['cluster_monthly_count']),
            'weighted_lng': np.average(df['county_lng'], weights=df['cluster_monthly_count']),
            'total_cluster_count': df['cluster_monthly_count'].sum(),
            'top_counties': ' -> '.join(
                df.sort_values('cluster_monthly_count', ascending=False)['county'].head(3)
            )
        }),
        include_groups=False
    )
    .reset_index(drop=True)
    .sort_values(['cluster', 'month'])
)

def describe_direction(lat_change, lng_change, threshold=0.15):
    north_south = ''
    east_west = ''

    if lat_change > threshold:
        north_south = 'north'
    elif lat_change < -threshold:
        north_south = 'south'

    if lng_change > threshold:
        east_west = 'east'
    elif lng_change < -threshold:
        east_west = 'west'

    if north_south and east_west:
        return f'{north_south}{east_west}'
    if north_south:
        return north_south
    if east_west:
        return east_west
    return 'stable'

movement_rows = []
for cluster_id, cluster_df in cluster_monthly_centers.groupby('cluster'):
    cluster_df = cluster_df.sort_values('month').reset_index(drop=True)
    for i in range(len(cluster_df) - 1):
        start_row = cluster_df.iloc[i]
        end_row = cluster_df.iloc[i + 1]
        movement_rows.append({
            'cluster': cluster_id,
            'start_month': int(start_row['month']),
            'end_month': int(end_row['month']),
            'start_counties': start_row['top_counties'],
            'end_counties': end_row['top_counties'],
            'lat_change': end_row['weighted_lat'] - start_row['weighted_lat'],
            'lng_change': end_row['weighted_lng'] - start_row['weighted_lng'],
            'direction': describe_direction(
                end_row['weighted_lat'] - start_row['weighted_lat'],
                end_row['weighted_lng'] - start_row['weighted_lng']
            )
        })

cluster_direction_steps = pd.DataFrame(movement_rows)

cluster_path_summary = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(
        first_active_month=('start_month', 'min'),
        last_active_month=('end_month', 'max'),
        representative_path=('start_counties', lambda s: ' | '.join(pd.Series(s).drop_duplicates().head(4))),
        dominant_directions=('direction', lambda s: ' -> '.join(pd.Series(s).replace('stable', np.nan).dropna().head(6)))
    )
)

overall_direction = (
    cluster_direction_steps
    .groupby('cluster', as_index=False)
    .agg(total_lat_change=('lat_change', 'sum'), total_lng_change=('lng_change', 'sum'))
)
overall_direction['overall_direction'] = overall_direction.apply(
    lambda row: describe_direction(row['total_lat_change'], row['total_lng_change'], threshold=0.3),
    axis=1
)

cluster_path_summary = cluster_path_summary.merge(overall_direction[['cluster', 'overall_direction']], on='cluster', how='left')
cluster_path_summary['dominant_directions'] = cluster_path_summary['dominant_directions'].replace('', 'stable')

month_names = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
    7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

# Function to convert numeric cluster metrics into plain-language traits
def classify_movement_traits(row):
    traits = []
    if row['avg_county_turnover'] >= 0.65:
        traits.append('high county turnover')
    elif row['avg_county_turnover'] >= 0.35:
        traits.append('moderate county turnover')
    else:
        traits.append('low county turnover')

    if row['active_months'] >= 9:
        traits.append('year-round or near year-round activity')
    elif row['active_months'] >= 5:
        traits.append('clear seasonal presence')
    else:
        traits.append('short or sporadic seasonal windows')

    if row['counties_visited'] >= 30:
        traits.append('broad statewide footprint')
    elif row['counties_visited'] >= 8:
        traits.append('regional multi-county footprint')
    else:
        traits.append('narrow county footprint')

    return ', '.join(traits)

# Use the path summary as a lookup table when writing the narrative for each cluster.
cluster_path_lookup = cluster_path_summary.set_index('cluster')
cluster_lines = ['### Cluster Interpretations']

# Render the final writeup
for _, summary_row in flight_cluster_summary.reset_index().sort_values('cluster').iterrows():
    cluster_num = int(summary_row['cluster'])
    display_cluster_num = cluster_num + 1
    path_row = cluster_path_lookup.loc[cluster_num]
    # Pull representative family examples and convert the metrics into plain-language traits.
    examples = ', '.join(f'**{family}**' for family in cluster_example_families.get(cluster_num, []))
    movement_traits = classify_movement_traits(summary_row)
    month_span = f"{month_names[int(path_row['first_active_month'])]} to {month_names[int(path_row['last_active_month'])]}"

    # Build one short paragraph block per cluster for the markdown report.
    cluster_lines.append(f"#### Cluster {display_cluster_num}")
    cluster_lines.append(
        f"This cluster contains **{int(summary_row['family_count'])} families** and is characterized by {movement_traits}. "
        f"On average, families in this cluster are active across **{summary_row['active_months']:.1f} months**, visit **{summary_row['counties_visited']:.1f} counties**, and show **{summary_row['avg_county_turnover']:.2f}** month-to-month county turnover."
    )
    cluster_lines.append(
        f"Across {month_span}, the representative county path is **{path_row['representative_path']}**. "
        f"The dominant month-to-month movement directions are **{path_row['dominant_directions']}**, and the overall drift is **{path_row['overall_direction']}**."
    )
    cluster_lines.append(f"Example families in this cluster include {examples}.")
    cluster_lines.append('')

display(Markdown('\n'.join(cluster_lines)))

### Cluster Interpretations
#### Cluster 1
This cluster contains **55 families** and is characterized by moderate county turnover, year-round or near year-round activity, broad statewide footprint. On average, families in this cluster are active across **11.6 months**, visit **55.5 counties**, and show **0.36** month-to-month county turnover.
Across January to December, the representative county path is **Kit Carson -> Pueblo -> Baca | Bent -> Boulder -> Mesa | Rio Grande -> Conejos -> Larimer | Larimer -> Boulder -> Lincoln**. The dominant month-to-month movement directions are **west -> west -> northeast -> west -> east -> northeast**, and the overall drift is **northwest**.
Example families in this cluster include **Ducks, Geese, and Waterfowl**, **Crows, Jays, and Magpies**, **Troupials and Allies**, **New World Sparrows**, **Pheasants, Grouse, and Allies**.

#### Cluster 2
This cluster contains **10 families** and is characterized by moderate county turnover, short or sporadic seasonal windows, narrow county footprint. On average, families in this cluster are active across **3.6 months**, visit **4.0 counties**, and show **0.46** month-to-month county turnover.
Across January to December, the representative county path is **Mesa -> Boulder | Larimer -> Denver | Boulder -> Larimer | Boulder -> Larimer -> Huerfano**. The dominant month-to-month movement directions are **northeast -> southwest -> east -> northwest -> southeast -> northwest**, and the overall drift is **southeast**.
Example families in this cluster include **Guineafowl**, **Skuas and Jaegers**, **Old World Parrots**, **Anhingas**, **Storks**.


## Model Evaluation

In [19]:
from IPython.display import Markdown, display

evaluation_lines = [
    '### Evaluation Results',
    '',
    f"- **Bird family prevalence over time:** The selected cluster count was **{prevalence_best_k}**. The **Silhouette Score** was **{prevalence_metrics['silhouette_score']:.4f}**, and the **Davies-Bouldin Index** was **{prevalence_metrics['davies_bouldin_index']:.4f}**.",
    f"- **Flight-pattern types by family:** The selected cluster count was **{flight_best_k}**. The **Silhouette Score** was **{flight_metrics['silhouette_score']:.4f}**, and the **Davies-Bouldin Index** was **{flight_metrics['davies_bouldin_index']:.4f}**.",
    '',
    'Higher Silhouette Scores indicate better-separated clusters, while lower Davies-Bouldin values indicate tighter and more distinct cluster structure.'
]

display(Markdown('\n'.join(evaluation_lines)))

### Evaluation Results

- **Bird family prevalence over time:** The selected cluster count was **2**. The **Silhouette Score** was **0.5151**, and the **Davies-Bouldin Index** was **0.7198**.
- **Flight-pattern types by family:** The selected cluster count was **2**. The **Silhouette Score** was **0.5985**, and the **Davies-Bouldin Index** was **0.7432**.

Higher Silhouette Scores indicate better-separated clusters, while lower Davies-Bouldin values indicate tighter and more distinct cluster structure.

## Challenges and Solutions

- **Challenge:** Bird counts span very different scales across counties and families.  
  **Solution:** Standardization and `log1p` transforms were used to keep large counts from dominating Euclidean distance.

- **Challenge:** Flight paths are not directly observed, and the cleaned modeling table is aggregated at the county-month level rather than as individual tracked trajectories.  
  **Solution:** County centroids were estimated from the raw eBird latitude and longitude observations in `ebird_co.csv`, then monthly cluster centers were computed from county-level bird counts to infer likely corridors and overall movement direction.

- **Challenge:** Choosing `k` is subjective in unsupervised learning.  
  **Solution:** Multiple values of `k` were compared using both Silhouette Score and Davies-Bouldin Index.

## Final Interpretation Notes

- Higher Silhouette Score is better because it indicates tighter, better-separated clusters.
- Lower Davies-Bouldin Index is better because it indicates lower within-cluster dispersion relative to between-cluster separation.
- If either metric is weak for a task, the clusters may still be useful for exploration, but they should be interpreted as soft groupings rather than definitive ecological classes.